In [ ]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…

In [ ]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### ERA5 Monthly Aggregated
#### Selected temp dataset

In [ ]:
# Load ERA5-Land monthly aggregated dataset and filter by date
dataset = (
    ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
    .filterDate('2023-01-01', '2023-02-01')
    .first()
)

# Clip to Panama
temperature = dataset.clip(panama_geom)

# Visualization parameters
visualization = {
    'bands': ['temperature_2m'],
    'min': 250,
    'max': 320,
    'palette': [
        '000080', '0000d9', '4000ff', '8000ff',
        '0080ff', '00ffff', '00ff80', '80ff00',
        'daff00', 'ffff00', 'fff500', 'ffda00',
        'ffb000', 'ffa400', 'ff4f00', 'ff2500',
        'ff0a00', 'ff00ff',
    ]
}

# Add clipped temperature layer
Map.add_layer(
    temperature,
    visualization,
    'Air temperature [K] at 2m height',
    True,
    0.8
)

### ERA5 ECMWF Climate Reanalysis

In [ ]:
# Load ERA5 hourly dataset
dataset = (
    ee.ImageCollection('ECMWF/ERA5/HOURLY')
    .filter(ee.Filter.date('2020-07-01', '2020-07-02'))
)

# Clip each image to Panama
temperature = dataset.map(lambda img: img.clip(panama_geom))

# Visualization parameters
visualization = {
    'bands': ['temperature_2m'],
    'min': 250.0,
    'max': 320.0,
    'palette': [
        '000080', '0000d9', '4000ff', '8000ff',
        '0080ff', '00ffff', '00ff80', '80ff00',
        'daff00', 'ffff00', 'fff500', 'ffda00',
        'ffb000', 'ffa400', 'ff4f00', 'ff2500',
        'ff0a00', 'ff00ff',
    ]
}

# Add clipped ERA5 layer
Map.add_layer(
    temperature,
    visualization,
    'Air temperature [K] at 2m height'
)

### TerraClimate Monthly

In [ ]:
# Load TerraClimate dataset
dataset = (
    ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
    .filter(ee.Filter.date('2017-07-01', '2017-08-01'))
)

# Select maximum temperature band and clip each image to Panama
maximum_temperature = dataset.select('tmmx').map(
    lambda img: img.clip(panama_geom)
)

# Visualization parameters
maximum_temperature_vis = {
    'min': -300.0,
    'max': 300.0,
    'palette': [
        '1a3678',
        '2955bc',
        '5699ff',
        '8dbae9',
        'acd1ff',
        'caebff',
        'e5f9ff',
        'fdffb4',
        'ffe6a2',
        'ffc969',
        'ffa12d',
        'ff7c1f',
        'ca531a',
        'ff0000',
        'ab0000',
    ],
}

# Add clipped temperature layer
Map.add_layer(
    maximum_temperature,
    maximum_temperature_vis,
    'Maximum Temperature'
)

### CPC

In [ ]:
# Load NOAA CPC Temperature dataset
dataset = (
    ee.ImageCollection('NOAA/CPC/Temperature')
    .filter(ee.Filter.date('2018-01-01', '2019-01-01'))
)

# Select maximum temperature band and clip each image to Panama
temperature = dataset.select('tmax').map(
    lambda img: img.clip(panama_geom)
)

# Visualization parameters
temperature_vis = {
    'min': -40,
    'max': 50,
    'palette': [
        '#ADD8E6',  # light blue
        '#008000',  # green
        '#FFFF00',  # yellow
        '#FFA500',  # orange
        '#FF0000',  # red
        '#800080'   # purple
    ],
}

# Add clipped temperature layer
Map.add_layer(
    temperature,
    temperature_vis,
    'Panama NOAA CPC Tmax'
)

In [ ]:
# Display map
Map.centerObject(panama_geom, 7)
Map